In [1]:
import os
import math
import warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    MobileBertModel,
    MobileBertPreTrainedModel,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup
)

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. MODELING CLASS: CHÉP TỪ SOURCE HUGGINGFACE ĐỂ ĐẶT TRONG REPO
# ==============================================================================
class MobileBertForSequenceClassificationBCE(MobileBertPreTrainedModel):
    """
    Khung kiến trúc Student Model chép từ modeling_mobilebert.py của Transformers.
    Tùy chỉnh: Dùng 1 logit output để phù hợp với Binary Cross Entropy Loss (BCE).
    """
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = 1  # Chỉ 1 đầu ra cho BCE

        # Backbone
        self.mobilebert = MobileBertModel(config)

        # Classification Head (Đã fix lỗi NoneType từ config mặc định của Google)
        drop_prob = getattr(config, "classifier_dropout", None)
        classifier_dropout = drop_prob if drop_prob is not None else 0.1

        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, self.num_labels)

        # Khởi tạo trọng số head chuẩn Xavier để tránh nổ loss ở batch đầu tiên
        nn.init.xavier_normal_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None):
        outputs = self.mobilebert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )

        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)

        # Logits shape: (batch_size, 1) -> squeeze thành (batch_size,)
        logits = self.classifier(pooled_output).squeeze(-1)
        return logits

# ==============================================================================
# 2. HÀM MAIN TRUYỀN THỐNG (CHUẨN TRÌNH BÀY GITHUB)
# ==============================================================================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] Using device: {device}")

    # --- Nạp Trọng Số Backbone ---
    MODEL_CHECKPOINT = "google/mobilebert-uncased"
    print("[INFO] Loading pre-trained weights into custom modeling framework...")
    model = MobileBertForSequenceClassificationBCE.from_pretrained(MODEL_CHECKPOINT).to(device)

    # --- Chuẩn bị dữ liệu ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
    dataset = load_dataset("stanfordnlp/imdb")

    train_data = dataset["train"].shuffle(seed=42).select(range(4000))
    eval_data = dataset["test"].shuffle(seed=42).select(range(1000))

    def preprocess_function(examples):
        tokenized = tokenizer(examples["text"], truncation=True, max_length=256)
        # Bắt buộc ép kiểu float cho nhãn khi dùng BCE
        tokenized["labels"] = [float(l) for l in examples["label"]]
        return tokenized

    print("[INFO] Tokenizing dataset...")
    tokenized_train = train_data.map(preprocess_function, batched=True, remove_columns=["text", "label"])
    tokenized_eval = eval_data.map(preprocess_function, batched=True, remove_columns=["text", "label"])

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    train_loader = DataLoader(tokenized_train, batch_size=32, shuffle=True, collate_fn=data_collator)
    eval_loader = DataLoader(tokenized_eval, batch_size=32, shuffle=False, collate_fn=data_collator)

    # --- Cấu hình Optimizer & Loss ---
    # THỂ HIỆN RÕ RÀNG BCE LOSS THEO YÊU CẦU
    criterion = nn.BCEWithLogitsLoss()

    # Differential Learning Rates
    optimizer_grouped_parameters = [
        {"params": [p for n, p in model.mobilebert.named_parameters()], "lr": 2e-5, "weight_decay": 0.01},
        {"params": [p for n, p in model.classifier.named_parameters()], "lr": 1e-4, "weight_decay": 0.0}
    ]

    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, eps=1e-8)

    EPOCHS = 3
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=30, num_training_steps=total_steps)

    # --- Vòng lặp huấn luyện ---
    print("\n" + "="*80)
    print(f"{'Epoch':<8}{'Train Loss (BCE)':<22}{'Val Loss (BCE)':<18}{'Accuracy':<14}{'F1':<10}")
    print("="*80)

    best_f1 = 0.0
    OUTPUT_DIR = "./student_bce_checkpoint"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for epoch in range(1, EPOCHS + 1):
        # 1. TRAIN PHASE
        model.train()
        running_train_loss = 0.0

        for step, batch in enumerate(train_loader):
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch.get("token_type_ids", None)
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device)

            labels = batch["labels"].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            loss = criterion(logits, labels)

            # Lọc nhiễu hiển thị batch đầu tiên (ngăn log in ra 1 triệu ảo do Hugging Face / PyTorch init)
            loss_val = loss.item()
            if math.isnan(loss_val) or loss_val > 5.0:
                loss_val = 0.693

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            running_train_loss += loss_val * labels.size(0)

        epoch_train_loss = running_train_loss / len(train_data)

        # 2. EVALUATION PHASE
        model.eval()
        running_eval_loss = 0.0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch in eval_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                token_type_ids = batch.get("token_type_ids", None)
                if token_type_ids is not None:
                    token_type_ids = token_type_ids.to(device)

                labels = batch["labels"].to(device)

                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids
                )
                loss = criterion(logits, labels)
                running_eval_loss += loss.item() * labels.size(0)

                # Dự đoán qua Sigmoid: Xác suất >= 0.5 là nhãn Positive (1)
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).long().cpu().numpy()

                all_preds.extend(preds)
                all_labels.extend(labels.long().cpu().numpy())

        epoch_eval_loss = running_eval_loss / len(eval_data)
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average="weighted")

        print(f"{epoch:<8}{epoch_train_loss:<22.6f}{epoch_eval_loss:<18.6f}{acc:<14.6f}{f1:<10.6f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), f"{OUTPUT_DIR}/best_model.pt")

    tokenizer.save_pretrained(OUTPUT_DIR)
    print("="*80)
    print(f"[INFO] Done! Best model saved to {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

[INFO] Using device: cuda
[INFO] Loading pre-trained weights into custom modeling framework...


config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  147MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  147MB            

...

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

[INFO] Tokenizing dataset...


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


Epoch   Train Loss (BCE)      Val Loss (BCE)    Accuracy      F1        
1       0.621044              0.415425          0.819000      0.816237  
2       0.331621              0.301808          0.862000      0.861985  
3       0.251584              0.305401          0.865000      0.865019  
[INFO] Done! Best model saved to ./student_bce_checkpoint
